# 🔍 Explainability Suite for DeiT-Tiny Model

## Applying LIME, SHAP, and Grad-CAM on `best_deit_tiny_patch16_224.pth`

---

This notebook walks through three state-of-the-art **model explainability** techniques:

| Technique | Type | What it answers |
|-----------|------|-----------------|
| **Grad-CAM** | Gradient-based | *Which spatial regions did the model focus on?* |
| **LIME** | Perturbation-based | *Which image segments positively contributed to the prediction?* |
| **SHAP** | Game-theoretic | *How much did each pixel contribute to the output score?* |

---

### Architecture Note: DeiT-Tiny
DeiT (Data-efficient Image Transformers) is a Vision Transformer variant. Unlike CNNs:
- It has **no convolutional layers** — images are split into 16×16 patches
- It uses **self-attention** across 196 patch tokens + 1 class token
- Grad-CAM requires special handling since there are no feature maps — we hook onto patch activations


## 📦 Step 1 — Install Dependencies

In [ ]:
# Install all required libraries
# - timm: PyTorch Image Models (DeiT lives here)
# - lime: Local Interpretable Model-agnostic Explanations
# - shap: SHapley Additive exPlanations
# - opencv-python: for heatmap colorization

!pip install timm lime shap opencv-python matplotlib scikit-image -q

## 📚 Step 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

import timm
from lime import lime_image
from skimage.segmentation import mark_boundaries
import shap

print(f"PyTorch version : {torch.__version__}")
print(f"Device available: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## ⚙️ Step 3 — Configuration

Set your paths and class names here.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

MODEL_PATH  = 'best_deit_tiny_patch16_224.pth'   # Path to your saved checkpoint
IMAGE_PATH  = 'test_image.jpg'                   # Path to the image to explain
NUM_CLASSES = 10                                  # Number of classes your model was trained on

# Optional: provide human-readable class names (must match NUM_CLASSES length)
# Leave as None to just show class indices
CLASS_NAMES = None
# Example:
# CLASS_NAMES = ['cat', 'dog', 'bird', 'car', 'truck',
#                'ship', 'plane', 'deer', 'frog', 'horse']

# LIME settings
LIME_NUM_SAMPLES  = 1000   # More samples = slower but more accurate
LIME_NUM_FEATURES = 10     # Number of superpixel regions to highlight

# SHAP settings
SHAP_BACKGROUND_SIZE = 20  # Number of background samples for GradientExplainer

# ─────────────────────────────────────────────────────────────────────────────
print('Configuration set!')

## 🏗️ Step 4 — Load the DeiT-Tiny Model

We use `timm` to recreate the DeiT-Tiny architecture and load your saved weights.

**Why `strict=False`?**  
Some checkpoints include optimizer states or extra keys. `strict=False` loads only matching keys and skips the rest safely.

In [ ]:
def load_model(model_path, num_classes, device):
    """
    Loads a DeiT-Tiny model from a .pth checkpoint.
    Handles multiple checkpoint formats automatically.
    """
    # Step 1: Create the DeiT-Tiny architecture with timm
    # pretrained=False because we're loading our own weights
    model = timm.create_model(
        'deit_tiny_patch16_224',
        pretrained=False,
        num_classes=num_classes
    )

    # Step 2: Load the checkpoint file
    checkpoint = torch.load(model_path, map_location=device)

    # Step 3: Extract state_dict — handles different saving conventions
    if isinstance(checkpoint, dict):
        state_dict = (
            checkpoint.get('model_state_dict') or   # common training loop format
            checkpoint.get('state_dict')        or   # Lightning / Keras-style
            checkpoint.get('model')             or   # some custom formats
            checkpoint                               # raw state_dict
        )
    else:
        state_dict = checkpoint  # already a state_dict object

    # Step 4: Strip 'module.' prefix added by DataParallel training
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

    # Step 5: Load weights (strict=False to ignore mismatched keys)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'  ⚠ Missing keys   : {len(missing)}')
    if unexpected:
        print(f'  ⚠ Unexpected keys: {len(unexpected)}')

    model.to(device)
    model.eval()  # Switch to inference mode (disables dropout, etc.)
    return model


model = load_model(MODEL_PATH, NUM_CLASSES, DEVICE)
print(f'✅ Model loaded successfully on {DEVICE}')

# Quick architecture summary
total_params = sum(p.numel() for p in model.parameters())
print(f'   Total parameters : {total_params:,}')
print(f'   Patch size       : 16×16')
print(f'   Number of patches: 196 (14×14 grid for 224×224 image)')

## 🖼️ Step 5 — Load & Preprocess the Image

DeiT expects:
- Size: **224 × 224**
- Normalized with **ImageNet mean and std** (since DeiT was pretrained on ImageNet)

We keep both the original numpy array (for visualization) and the normalized tensor (for the model).

In [ ]:
# ImageNet normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def load_image(image_path):
    """
    Returns:
      pil_img : PIL Image (RGB, 224×224)          — for LIME
      np_img  : numpy uint8 array (H, W, 3)       — for visualization & Grad-CAM overlay
      tensor  : normalized torch tensor (1,3,H,W) — for model inference
    """
    pil_img = Image.open(image_path).convert('RGB').resize((224, 224))
    np_img  = np.array(pil_img)                   # shape: (224, 224, 3), dtype: uint8
    tensor  = preprocess(pil_img).unsqueeze(0)    # shape: (1, 3, 224, 224)
    return pil_img, np_img, tensor


pil_img, np_img, tensor = load_image(IMAGE_PATH)
tensor = tensor.to(DEVICE)

print(f'Image shape (numpy) : {np_img.shape}  dtype: {np_img.dtype}')
print(f'Tensor shape        : {tensor.shape}  dtype: {tensor.dtype}')

# Display the original image
plt.figure(figsize=(4, 4))
plt.imshow(np_img)
plt.title('Input Image', fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 🧠 Step 6 — Run Inference & Get Top Predictions

`torch.no_grad()` disables gradient tracking during inference — saves memory and speeds things up.

In [ ]:
def get_predictions(model, tensor, class_names=None, top_k=5):
    """
    Runs inference and returns top-k predictions with probabilities.
    top_k is automatically clamped to NUM_CLASSES so topk() never
    raises 'selected index k out of range' when the model has fewer
    than 5 output classes.
    """
    with torch.no_grad():
        logits = model(tensor)                              # raw scores
        probs  = F.softmax(logits, dim=1).cpu()            # convert to probabilities

    # --- FIX: clamp k so it never exceeds the actual number of classes ---
    num_classes_actual = probs.shape[1]
    safe_k = min(top_k, num_classes_actual)
    if safe_k < top_k:
        print(f'  ℹ️  top_k clamped from {top_k} → {safe_k} (model only has {num_classes_actual} classes)')

    top = probs.topk(safe_k, dim=1)
    indices = top.indices[0].cpu().numpy()
    values  = top.values[0].cpu().numpy()

    results = []
    for idx, prob in zip(indices, values):
        name = class_names[idx] if class_names and idx < len(class_names) else f'Class {idx}'
        results.append((int(idx), name, float(prob)))
    return results


predictions = get_predictions(model, tensor, CLASS_NAMES)
top_class   = predictions[0][0]  # index of highest-confidence class
top_label   = predictions[0][1]

print('='*50)
print(f'Top-{len(predictions)} Predictions')
print('='*50)
for rank, (idx, name, prob) in enumerate(predictions, 1):
    bar = '█' * int(prob * 40)
    print(f'{rank}. [{idx:4d}] {name:30s} {prob*100:6.2f}%  {bar}')

print(f'\n➡ Explaining prediction: "{top_label}" (class {top_class})')

---

# 🔥 Part A — Grad-CAM

## Theory
**Gradient-weighted Class Activation Mapping (Grad-CAM)** uses the gradients flowing into the final feature representation to understand which spatial locations were most important for a prediction.

### How it works for DeiT (Vision Transformer):
1. Forward pass → hook saves **patch token activations** from the last transformer block
2. Backpropagate the target class score → hook saves **gradients** at same layer
3. Compute importance: `weight = mean(gradient over embed_dim)` per patch
4. Weighted sum of activations → reshape 14×14 → upsample to 224×224
5. Apply ReLU (keep only positive influences) and normalize

> **Note:** For CNNs, Grad-CAM hooks on the last conv layer. For ViTs, we hook on the last transformer block's `norm1` layer, which captures rich spatial patch representations.

In [ ]:
class GradCAM:
    """
    Grad-CAM implementation for Vision Transformers (DeiT).

    Key insight: DeiT processes images as sequences of 196 patch tokens.
    We compute gradient-weighted activations over these patch tokens
    and reshape them back into a 14×14 spatial grid.
    """

    def __init__(self, model):
        self.model      = model
        self.gradients  = None   # will store gradients from backward hook
        self.activations = None  # will store features from forward hook
        self._register_hooks()

    def _register_hooks(self):
        """
        Registers forward and backward hooks on the last transformer block.
        Hooks are callback functions called automatically during forward/backward pass.
        """
        # Target: last block's LayerNorm (applied before the attention sub-layer)
        # This captures rich, class-discriminative spatial information
        target_layer = self.model.blocks[-1].norm1

        def forward_hook(module, input, output):
            """Captures the activations (output) during forward pass."""
            self.activations = output.detach().cpu()  # shape: (1, N, C) — moved to CPU immediately

        def backward_hook(module, grad_in, grad_out):
            """Captures gradients flowing back through this layer."""
            self.gradients = grad_out[0].detach().cpu()  # shape: (1, N, C) — moved to CPU immediately

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_full_backward_hook(backward_hook)

    def generate_cam(self, tensor, class_idx=None):
        """
        Generate the Grad-CAM heatmap.

        Args:
            tensor    : Input image tensor (1, 3, 224, 224)
            class_idx : Target class to explain. None = use predicted class.

        Returns:
            cam       : Normalized heatmap (224, 224) in range [0, 1]
            class_idx : The class that was explained
        """
        self.model.zero_grad()

        # Forward pass — triggers forward hook
        output = self.model(tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Compute gradient of the target class score w.r.t. activations
        # This triggers the backward hook
        score = output[0, class_idx]
        score.backward()

        # --- Core Grad-CAM computation ---
        # activations & gradients shape: (1, N, C)
        # N = 197 (1 class token + 196 patch tokens), C = 192 (embed dim)
        grads = self.gradients[0]    # (N, C)
        acts  = self.activations[0]  # (N, C)

        # Remove class token (index 0) — we only care about spatial patches
        grads = grads[1:]  # (196, C)
        acts  = acts[1:]   # (196, C)

        # Importance weight per patch = mean gradient magnitude across embed dim
        weights = grads.mean(dim=-1)            # (196,)

        # Weighted sum of activations to get per-patch importance score
        cam = (weights.unsqueeze(-1) * acts).sum(dim=-1)   # (196,)

        # Apply ReLU: keep only features that positively influence the class
        # cam is a CPU tensor (hooks already called .cpu()); convert safely
        cam = F.relu(cam.clone()).cpu().numpy()

        # Reshape from flat 196 → 14×14 spatial grid
        grid_size = int(cam.shape[0] ** 0.5)   # sqrt(196) = 14
        cam = cam.reshape(grid_size, grid_size)

        # Upsample to original image size (224×224)
        cam = cv2.resize(cam, (224, 224))

        # Normalize to [0, 1] for visualization
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        return cam, class_idx

    def overlay_heatmap(self, np_img, cam, alpha=0.5, colormap=cv2.COLORMAP_JET):
        """
        Overlays the Grad-CAM heatmap on the original image.

        Args:
            np_img  : Original image (H, W, 3) uint8
            cam     : Grad-CAM heatmap (H, W) in [0, 1]
            alpha   : Heatmap opacity (0=original, 1=full heatmap)
            colormap: OpenCV colormap (JET=blue-green-red spectrum)
        """
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), colormap)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)  # OpenCV uses BGR by default
        overlay = (alpha * heatmap + (1 - alpha) * np_img).astype(np.uint8)
        return overlay


print('GradCAM class defined ✅')

In [ ]:
# Run Grad-CAM
print('Running Grad-CAM...')

gradcam = GradCAM(model)

# We need gradients — clone tensor and enable gradient tracking
input_tensor = tensor.clone().requires_grad_(True)

cam, cam_class = gradcam.generate_cam(input_tensor, class_idx=top_class)
gradcam_overlay = gradcam.overlay_heatmap(np_img, cam, alpha=0.5)

print(f'✅ Grad-CAM generated for class: {top_label} (index {cam_class})')

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Grad-CAM Analysis', fontsize=15, fontweight='bold')

axes[0].imshow(np_img)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

im = axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap (raw)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(gradcam_overlay)
axes[2].set_title(f'Grad-CAM Overlay\nPredicted: {top_label}', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('gradcam_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gradcam_result.png')

### 📖 Interpreting Grad-CAM
- **Red/warm areas** → regions the model found most important for this prediction
- **Blue/cool areas** → regions that had little influence
- A good model should highlight semantically meaningful regions (e.g., the animal's face, a car's body)

---

# 🟩 Part B — LIME

## Theory
**LIME (Local Interpretable Model-agnostic Explanations)** explains *any* black-box model by learning a simple, interpretable model locally around a single prediction.

### How it works:
1. **Segment** the image into superpixels (coherent color regions)
2. **Perturb**: generate `N` variants by randomly turning superpixels ON/OFF
3. **Query**: run each variant through the black-box model to get predictions
4. **Fit**: train a weighted linear model where closer perturbations get higher weights
5. **Explain**: the linear model's coefficients tell us which superpixels mattered most

### Pros vs Cons:
✅ Model-agnostic — works with any model  
✅ Human-interpretable regions  
⚠️ Computationally expensive (runs N forward passes)  
⚠️ Non-deterministic (uses random perturbations)

In [ ]:
def make_lime_predictor(model, device):
    """
    Creates a prediction function compatible with LIME's interface.

    LIME requires a function: list of HWC uint8 images → probability array
    We wrap our PyTorch model in this format.
    """
    def batch_predict(images_np):
        """
        Args:
            images_np: list of numpy arrays, each shape (224, 224, 3), dtype uint8
        Returns:
            probs: numpy array of shape (N, num_classes)
        """
        batch = []
        for img in images_np:
            pil = Image.fromarray(img.astype(np.uint8))
            batch.append(preprocess(pil))   # normalize and convert to tensor

        batch_tensor = torch.stack(batch).to(device)    # (N, 3, 224, 224)

        with torch.no_grad():
            logits = model(batch_tensor)
            probs  = F.softmax(logits, dim=1).cpu().numpy()

        return probs   # shape: (N, num_classes)

    return batch_predict


print('LIME predictor function defined ✅')

In [ ]:
# Run LIME
print(f'Running LIME with {LIME_NUM_SAMPLES} samples...')
print('(This runs the model hundreds of times — may take 1–3 minutes)')

batch_predict = make_lime_predictor(model, DEVICE)

# Create LIME explainer
lime_explainer = lime_image.LimeImageExplainer()

# Generate explanation
# - top_labels=5: explain the top 5 predicted classes
# - hide_color=0: masked superpixels are shown as black (0)
# - num_samples: how many perturbed images to generate
# FIX: clamp top_labels so it never exceeds NUM_CLASSES
safe_top_labels = min(5, NUM_CLASSES)
lime_explanation = lime_explainer.explain_instance(
    np_img,
    batch_predict,
    top_labels=safe_top_labels,
    hide_color=0,
    num_samples=LIME_NUM_SAMPLES,
    random_seed=42,
)

lime_top_label = lime_explanation.top_labels[0]
print(f'✅ LIME explanation generated for label index: {lime_top_label}')

In [ ]:
# Extract and visualize LIME results

# positive_only=True  → show only regions that SUPPORT the prediction
# positive_only=False → show both supporting and opposing regions
# hide_rest=False     → show the full image (not just highlighted regions)

# --- Positive regions (what supports the prediction) ---
# FIX: clamp num_features to segments actually found by LIME
num_segments_found = len(lime_explanation.local_exp[lime_top_label])
safe_num_features = min(LIME_NUM_FEATURES, num_segments_found)
temp_pos, mask_pos = lime_explanation.get_image_and_mask(
    lime_top_label,
    positive_only=True,
    num_features=safe_num_features,
    hide_rest=False,
)
boundary_pos = mark_boundaries(temp_pos / 255.0, mask_pos)

# --- Both positive and negative regions ---
temp_both, mask_both = lime_explanation.get_image_and_mask(
    lime_top_label,
    positive_only=False,
    num_features=safe_num_features,
    hide_rest=False,
)
boundary_both = mark_boundaries(temp_both / 255.0, mask_both)

# --- Hidden rest: show only important regions ---
temp_hidden, mask_hidden = lime_explanation.get_image_and_mask(
    lime_top_label,
    positive_only=True,
    num_features=safe_num_features,
    hide_rest=True,
)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('LIME Explanation', fontsize=15, fontweight='bold')

axes[0].imshow(np_img)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(boundary_pos)
axes[1].set_title('Positive Regions Only\n(supports prediction)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(boundary_both)
axes[2].set_title('Positive + Negative Regions\n(green=for, red=against)', fontsize=12)
axes[2].axis('off')

axes[3].imshow(temp_hidden)
axes[3].set_title('Important Regions Only\n(rest hidden)', fontsize=12)
axes[3].axis('off')

plt.tight_layout()
plt.savefig('lime_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lime_result.png')

In [ ]:
# LIME: Show feature importance as a bar chart

# Get the local explanation weights (superpixel index → importance score)
lime_weights = lime_explanation.local_exp[lime_top_label]
lime_weights_sorted = sorted(lime_weights, key=lambda x: abs(x[1]), reverse=True)[:LIME_NUM_FEATURES]

labels  = [f'Segment {idx}' for idx, _ in lime_weights_sorted]
weights = [w for _, w in lime_weights_sorted]
colors  = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

plt.figure(figsize=(10, 5))
bars = plt.barh(labels, weights, color=colors, edgecolor='white', linewidth=0.5)
plt.xlabel('LIME Weight (positive = supports prediction)', fontsize=11)
plt.title(f'LIME Feature Importance — "{top_label}"', fontsize=13, fontweight='bold')
plt.axvline(0, color='white', linewidth=0.8, linestyle='--')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('lime_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lime_importance.png')

### 📖 Interpreting LIME
- **Green regions** → superpixels that *increase* the model's confidence in the predicted class
- **Red regions** → superpixels that *decrease* confidence (push toward other classes)
- The bar chart shows the importance weight of each segment — larger magnitude = more influential

---

# 🔵 Part C — SHAP (GradientExplainer)

## Theory
**SHAP (SHapley Additive exPlanations)** is rooted in cooperative game theory. It computes the *Shapley value* for each feature — the average marginal contribution of that feature across all possible coalitions.

### GradientExplainer specifically:
- Combines **SHAP values** with **Integrated Gradients**
- Uses a background dataset to define the "baseline" (what the model sees with no signal)
- Computes expected gradients over the background to estimate attribution
- Returns per-pixel, per-channel attributions: shape `(1, 3, 224, 224)`

### Pros vs Cons:
✅ Theoretically grounded (satisfies axioms: efficiency, symmetry, dummy, additivity)  
✅ Pixel-level granularity (finer than LIME's superpixels)  
✅ Faster than KernelExplainer for deep learning  
⚠️ Background distribution matters — we use Gaussian noise as a neutral baseline

In [ ]:
# Run SHAP GradientExplainer
print(f'Running SHAP GradientExplainer (background size={SHAP_BACKGROUND_SIZE})...')

# Background dataset: random Gaussian noise tensors
# These represent "neutral" images — a baseline with no meaningful signal
# The explainer will compute attributions relative to this baseline
background = torch.randn(SHAP_BACKGROUND_SIZE, 3, 224, 224).to(DEVICE)

# Create GradientExplainer
# - model   : the neural network
# - background: reference distribution ("what the model sees as neutral")
shap_explainer = shap.GradientExplainer(model, background)

# Compute SHAP values for our input image
# Returns: list of arrays, one per output class (or a single array)
# Each array shape: (1, 3, 224, 224) — attribution per pixel per channel
shap_values = shap_explainer.shap_values(tensor.to(DEVICE))

print(f'✅ SHAP values computed')
if isinstance(shap_values, list):
    print(f'   Number of classes explained : {len(shap_values)}')
    print(f'   Shape per class             : {shap_values[0].shape}')
else:
    print(f'   SHAP values shape: {shap_values.shape}')

In [ ]:
def to_numpy(x):
    """
    Safely convert a value to a numpy array regardless of whether it is
    already a numpy array, a CPU tensor, or a CUDA tensor.
    """
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)   # fallback for lists, etc.


def process_shap_values(shap_values, class_idx):
    """
    Convert raw SHAP values to visualization-ready heatmap.

    Steps:
    1. Extract SHAP values for the target class
    2. Convert to numpy safely (handles CUDA tensors from some SHAP versions)
    3. Average absolute values across RGB channels → single heatmap
    4. Normalize to [0, 1]

    Also returns a signed version showing positive vs negative contributions.
    """
    # Extract values for target class
    if isinstance(shap_values, list):
        sv_raw = shap_values[class_idx][0]   # (3, H, W)  — may be CUDA tensor
    else:
        sv_raw = shap_values[0]              # (3, H, W)

    # FIX: convert to CPU numpy regardless of what SHAP returned
    sv = to_numpy(sv_raw)   # (3, H, W) numpy float

    # Absolute heatmap: how much each pixel contributed (ignoring direction)
    abs_heat = np.abs(sv).mean(axis=0)   # (H, W)
    abs_heat_norm = abs_heat / (abs_heat.max() + 1e-8)

    # Signed heatmap: positive=supports, negative=opposes
    signed_heat = sv.mean(axis=0)        # (H, W)
    # Normalize to [-1, 1]
    max_abs = np.abs(signed_heat).max() + 1e-8
    signed_heat_norm = signed_heat / max_abs

    return abs_heat_norm, signed_heat_norm, sv


shap_abs, shap_signed, shap_raw = process_shap_values(shap_values, top_class)

print(f'SHAP heatmap range (abs)   : [{shap_abs.min():.3f}, {shap_abs.max():.3f}]')
print(f'SHAP heatmap range (signed): [{shap_signed.min():.3f}, {shap_signed.max():.3f}]')

In [ ]:
# Visualize SHAP results
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('SHAP GradientExplainer Analysis', fontsize=15, fontweight='bold')

# 1. Original image
axes[0].imshow(np_img)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

# 2. Absolute SHAP heatmap
im2 = axes[1].imshow(shap_abs, cmap='viridis')
axes[1].set_title('SHAP Absolute Attribution\n(how much each pixel contributed)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046)

# 3. Signed SHAP heatmap (diverging colormap: blue=negative, red=positive)
im3 = axes[2].imshow(shap_signed, cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title('SHAP Signed Attribution\n(red=supports, blue=opposes)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046)

# 4. Overlay on original image
shap_color = plt.cm.viridis(shap_abs)[:, :, :3]
shap_color_uint8 = (shap_color * 255).astype(np.uint8)
shap_overlay = (0.55 * shap_color_uint8 + 0.45 * np_img).astype(np.uint8)
axes[3].imshow(shap_overlay)
axes[3].set_title('SHAP Overlay on Image', fontsize=12)
axes[3].axis('off')

plt.tight_layout()
plt.savefig('shap_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_result.png')

In [ ]:
# SHAP: Per-channel analysis (R, G, B contributions)

channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
channel_cmaps = ['Reds', 'Greens', 'Blues']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SHAP Per-Channel Attribution', fontsize=14, fontweight='bold')

for i, (name, cmap) in enumerate(zip(channel_names, channel_cmaps)):
    channel_shap = np.abs(shap_raw[i])   # absolute attribution for channel i
    channel_norm = channel_shap / (channel_shap.max() + 1e-8)
    im = axes[i].imshow(channel_norm, cmap=cmap)
    axes[i].set_title(name, fontsize=12)
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.tight_layout()
plt.savefig('shap_channels.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_channels.png')

### 📖 Interpreting SHAP
- **Bright yellow/green (viridis)** → pixels with highest attribution — critical for the prediction
- **Dark areas** → pixels with low or no attribution
- **Signed map (red/blue)**: red pixels *increase* the class score, blue pixels *decrease* it
- Per-channel views reveal *which color information* the model used

---

# 📊 Part D — Side-by-Side Comparison

Comparing all three methods helps identify consensus regions — areas highlighted by multiple techniques are most reliable.

In [ ]:
# Full comparison: all methods side by side

fig = plt.figure(figsize=(20, 10), facecolor='#1a1a2e')
fig.suptitle(
    f'Explainability Suite — DeiT-Tiny  |  Predicted: "{top_label}"',
    fontsize=16, color='white', fontweight='bold', y=0.98
)

gs = gridspec.GridSpec(2, 3, figure=fig,
                       wspace=0.05, hspace=0.15,
                       left=0.02, right=0.98,
                       top=0.92, bottom=0.02)

label_color = '#e0e0e0'

# --- Row 1 ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(np_img)
ax1.set_title('Original Image', color=label_color, fontsize=12, pad=8)
ax1.axis('off')

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(gradcam_overlay)
ax2.set_title('Grad-CAM\n(gradient-weighted patch activations)', color=label_color, fontsize=12, pad=8)
ax2.axis('off')

ax3 = fig.add_subplot(gs[0, 2])
ax3.imshow(boundary_pos)
ax3.set_title('LIME\n(positive superpixel regions)', color=label_color, fontsize=12, pad=8)
ax3.axis('off')

# --- Row 2 ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.imshow(shap_overlay)
ax4.set_title('SHAP\n(pixel-level gradient attributions)', color=label_color, fontsize=12, pad=8)
ax4.axis('off')

ax5 = fig.add_subplot(gs[1, 1])
# Composite: blend all three
shap_rgb = plt.cm.viridis(shap_abs)[:, :, :3]
cam_rgb  = plt.cm.jet(cam)[:, :, :3]
composite = (
    0.4 * np_img / 255.0 +
    0.3 * cam_rgb +
    0.3 * shap_rgb
)
composite = np.clip(composite, 0, 1)
ax5.imshow(composite)
ax5.set_title('Composite\n(Grad-CAM + SHAP blended)', color=label_color, fontsize=12, pad=8)
ax5.axis('off')

ax6 = fig.add_subplot(gs[1, 2])
# Consensus map: multiply Grad-CAM and SHAP for regions both agree on
consensus = cam * shap_abs
consensus /= consensus.max() + 1e-8
ax6.imshow(consensus, cmap='hot')
ax6.set_title('Consensus Map\n(Grad-CAM × SHAP agreement)', color=label_color, fontsize=12, pad=8)
ax6.axis('off')

plt.savefig('explainability_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Saved: explainability_comparison.png')

---

# 📋 Part E — Summary Report

In [ ]:
# Summary statistics

print('=' * 60)
print('EXPLAINABILITY REPORT — DeiT-Tiny')
print('=' * 60)

print(f'\nModel              : DeiT-Tiny (patch 16×16, input 224×224)')
print(f'Checkpoint         : {MODEL_PATH}')
print(f'Image              : {IMAGE_PATH}')
print(f'Device             : {DEVICE}')

print(f'\nTop Prediction     : {top_label} (class {top_class})')
print(f'Confidence         : {predictions[0][2]*100:.2f}%')

print('\nGrad-CAM Stats:')
print(f'  Target layer     : model.blocks[-1].norm1')
print(f'  CAM min/max      : {cam.min():.4f} / {cam.max():.4f}')
print(f'  High-activation  : {(cam > 0.7).sum()} / {cam.size} pixels ({(cam>0.7).mean()*100:.1f}%)')

print('\nLIME Stats:')
print(f'  Num samples      : {LIME_NUM_SAMPLES}')
print(f'  Num features     : {LIME_NUM_FEATURES}')
print(f'  Positive segments: {mask_pos.sum()} pixels ({mask_pos.mean()*100:.1f}% of image)')

print('\nSHAP Stats:')
print(f'  Background size  : {SHAP_BACKGROUND_SIZE}')
print(f'  Max attribution  : {shap_abs.max():.4f}')
print(f'  Mean attribution : {shap_abs.mean():.4f}')
print(f'  High-attribution : {(shap_abs > 0.5).sum()} / {shap_abs.size} pixels ({(shap_abs>0.5).mean()*100:.1f}%)')

print('\nOutput Files:')
for f in ['gradcam_result.png', 'lime_result.png', 'lime_importance.png',
          'shap_result.png', 'shap_channels.png', 'explainability_comparison.png']:
    print(f'  ✅ {f}')

print('\n' + '=' * 60)

---

## 🗂️ Quick Method Comparison

| Property | Grad-CAM | LIME | SHAP |
|----------|----------|------|------|
| **Granularity** | Patch-level (14×14) | Superpixel regions | Pixel-level (224×224) |
| **Speed** | Fast (~1 sec) | Slow (N forward passes) | Medium |
| **Model dependency** | ViT-specific hooks | Model-agnostic | Gradient-based |
| **Theoretical basis** | Gradients | Local linear model | Shapley values |
| **Best for** | Quick spatial intuition | Human-friendly regions | Precise pixel attribution |
| **Randomness** | Deterministic | Stochastic | Semi-deterministic |

### 💡 Key Takeaway
- Use **Grad-CAM** for fast, coarse spatial checks
- Use **LIME** when you want interpretable region explanations for non-technical stakeholders
- Use **SHAP** when you need fine-grained, theoretically-grounded pixel attributions
- The **Consensus Map** (Grad-CAM × SHAP) highlights regions both methods agree on — those are the most reliable